In [ ]:
"""
Evaluation for external_pred_corrected.csv (from correct_external_predictions.py)
-- the out-of-domain Metamorphosis test set, corrected version.

Runs the same four complementary views as evaluate_oof_corrected.ipynb, once
for ALL LANGUAGES POOLED and once per individual language, all in one pass:
  1. Token-level  - is the label right per token? (ignores span boundaries)
  2. Span-level   - is the full span (start, end, label) right? (seqeval strict)
  3. Span-boundary-only - did the model find the span at all, ignoring label?
  4. BIO-prefix accuracy - B / I / O accuracy regardless of the event label
plus a confusion matrix for each.

Each report is computed on its own subset (pooled, or one language) treated
as one corpus -- no per-sentence averaging.

NOTE on file structure: word-level, one row per word. Column names differ
slightly from the k-fold OOF file -- 'gold_label' here (not 'true_label'),
plus a 'language' column since this file spans multiple languages. Sentences
are reconstructed by grouping on (language, source_file, sentence_id),
ordered by word_index.
"""

from itertools import chain

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')

from seqeval.metrics import classification_report as seq_report
from seqeval.scheme import IOB2
from sklearn.metrics import (
    classification_report as sk_report,
    ConfusionMatrixDisplay,
    confusion_matrix,
)

# -- Load ----------------------------------------------------------------
path = "PATH"
MODEL_FILE_TAG = "mmBERT-base"  # e.g. "mmBERT-base" or "xlm-roberta-base" -- must match correct_external_predictions.py's --base_model_short_name
df_all = pd.read_csv(f"{path}/external_pred_corrected_{MODEL_FILE_TAG}.csv")
# columns: language, source_file, sentence_id, word_index, word,
#          gold_label, pred_label_raw, pred_label_corrected,
#          correction_applied, correct

PRED_COL = "pred_label_corrected"  # swap for 'pred_label_raw' to check uncorrected predictions
EVENT_LABELS = ["process", "stative_event", "change_of_state", "non_event"]

languages = sorted(df_all["language"].unique())
print(f"Loaded {len(df_all)} word-level rows, languages found: {languages}")
print(f"Correction applied: {df_all['correction_applied'].unique().tolist()}")


In [ ]:
# -- Helper functions, shared across pooled + every per-language run -------

def strip_bio(label):
    return 'O' if label == 'O' else label.split('-', 1)[1]

def strip_suffix(label):
    return "O" if label == "O" else label.split("-")[0]

def flat_to_bio_binary(labels):
    return ["O" if l == "O" else l.split("-")[0] + "-EVENT" for l in labels]

def safe_seq_report(gold_seqs, pred_seqs, **kwargs):
    """seqeval's classification_report crashes (ValueError: max() arg is empty)
    if a subset has literally zero labeled entities of any kind -- common for
    a single language on a short test document. Guard rather than let one
    thin language kill the whole run."""
    try:
        return seq_report(gold_seqs, pred_seqs, **kwargs)
    except ValueError:
        return "(no entities found in gold or predictions for this subset -- nothing to report)"

def evaluate_subset(df, label):
    """Runs all four views + confusion matrix for one subset (pooled, or one language)."""
    print("\n" + "#" * 70)
    print(f"# {label}")
    print("#" * 70)

    gold_seqs, pred_seqs = [], []
    for _, g in df.sort_values("word_index").groupby(["language", "source_file", "sentence_id"], sort=False):
        gold_seqs.append(g["gold_label"].tolist())
        pred_seqs.append(g[PRED_COL].tolist())

    bad = [i for i, (g, p) in enumerate(zip(gold_seqs, pred_seqs)) if len(g) != len(p)]
    if bad:
        print(f"WARNING: {len(bad)} sentences have mismatched gold/pred lengths -- check these.")
    print(f"{len(gold_seqs)} sentences, {len(df)} words")

    flat_gold = [strip_bio(l) for l in chain.from_iterable(gold_seqs)]
    flat_pred = [strip_bio(l) for l in chain.from_iterable(pred_seqs)]

    print("\n-- 1. TOKEN-LEVEL --")
    print(sk_report(flat_gold, flat_pred, labels=EVENT_LABELS + ["O"], zero_division=0, digits=3))

    print("-- 2. SPAN-LEVEL, seqeval default (conlleval-style, no scheme enforcement) --")
    print(safe_seq_report(gold_seqs, pred_seqs, zero_division=0, digits=3))

    gold_bin = [flat_to_bio_binary(seq) for seq in gold_seqs]
    pred_bin = [flat_to_bio_binary(seq) for seq in pred_seqs]
    print("-- 3. SPAN-BOUNDARY ONLY, seqeval default (all event types merged -> EVENT) --")
    print(safe_seq_report(gold_bin, pred_bin, zero_division=0, digits=3))

    gold_bio_flat = [strip_suffix(l) for l in chain.from_iterable(gold_seqs)]
    pred_bio_flat = [strip_suffix(l) for l in chain.from_iterable(pred_seqs)]
    print("-- 4. B / I / O PREFIX ACCURACY --")
    print(sk_report(gold_bio_flat, pred_bio_flat, labels=["B", "I", "O"], zero_division=0, digits=3))

    cm_labels = EVENT_LABELS + ["O"]
    cm = confusion_matrix(flat_gold, flat_pred, labels=cm_labels)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=cm_labels)
    fig, ax = plt.subplots(figsize=(6, 6))
    disp.plot(ax=ax, xticks_rotation=45, colorbar=False)
    plt.title(f"Token-level confusion matrix -- external (Metamorphosis)\n{label}")
    plt.tight_layout()
    tag = label.replace(" ", "_").replace("=", "")
    plt.savefig(f"confusion_matrix_external_{tag}.png", dpi=150)
    plt.show()


In [ ]:
evaluate_subset(df_all, "ALL LANGUAGES POOLED")


In [ ]:
for lang in languages:
    evaluate_subset(df_all[df_all["language"] == lang], f"LANGUAGE = {lang}")
